# basic infomap function (und + unw) 😭

In [2]:
import igraph as ig
import numpy as np

In [3]:
# shannon entropy
# H = - sum(p log2 p)

def entropy(probs):

    probs = np.array(probs)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

## map equation 

In [4]:
def map_equation(g, partition):

    # nr of edges
    m = g.ecount()

    # node visit probabilities p_i
    # p_i = degree(i) / (2m)
    degrees = np.array(g.degree())
    p = degrees / (2 * m)

    # nr of communities
    num_comms = max(partition) + 1

    # community visit probabilities p_c
    p_module = np.zeros(num_comms)
    for i, c in enumerate(partition):
        p_module[c] += p[i]

    # exit probabilities q_c
    q = np.zeros(num_comms)

    edges = g.get_edgelist()

    for (i, j) in edges:
        ci = partition[i]
        cj = partition[j]

        # edge crosses communities -> counts as exit 
        # if the edge connects two communities, exits from both
        if ci != cj:
            q[ci] += 1
            q[cj] += 1

    # normalize
    q = q / (2 * m)

    # total exit probability
    q_total = np.sum(q)

    # entropy between communities H(Q)
    if q_total > 0:
        H_Q = entropy(q / q_total)
    else:
        H_Q = 0

    # map equation
    L = q_total * H_Q

    # entropy within each community
    for c in range(num_comms):

        # nodes in community c
        nodes_in_c = [i for i in range(len(partition)) if partition[i] == c]

        # probabilities inside community: node probs + exit prob
        probs = list(p[nodes_in_c])
        probs.append(q[c])

        total = sum(probs)

        if total > 0:
            probs = [x / total for x in probs]
            H_c = entropy(probs)

            L += (p_module[c] + q[c]) * H_c

    return L

## test on lauras unweighted + undirected

In [8]:
# lauras graphs

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")

# partition from ground truth (attribute "community")
partition = [int(v["community"]) for v in g.vs]

# compute description length
L_custom = map_equation(g, partition)

# compare with igraph Infomap
communities = g.community_infomap()
L_igraph = communities.codelength

print("Custom L:", L_custom)
print("igraph L:", L_igraph)

Custom L: 3.4015829735877694
igraph L: 3.40158297358777
